In [34]:
library(tidytable)
library(tidyverse)

In [35]:
start_date = ymd("2021-03-01")
end_date = ymd("2026-04-30")

In [36]:
df_new = read_csv("../data/raw/stop_data_2024_current.csv")
# df_new

New names:
• `` -> `...58`
Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 10931 Columns: 58
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (40): LEA_Record_ID, StopDate, CallForService, WelfareWellnessCheck, In...
dbl   (9): PersonNum, StopDuration, Latitude, Longitude, BlockNumber, Highwa...
lgl   (8): OtherLocationDesc, IsK12School, SchoolName, StopForAStudent, Traf...
time  (1): StopTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [46]:
df_old = read_csv("../data/raw/stop_data_2020_2023.csv")
# df_old |>
#     arrange(LEA_RecordID) |>
#     head(n = 14) |>
#     select(LEA_RecordID, DateTimeOfStop, ActionsTaken)

Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 18090 Columns: 44
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (37): LEA_RecordID, DateTimeOfStop, RespToServiceCall, InfoBasedStop, Ty...
dbl  (6): PersonNum, DateTimeSortable, DurationOfStop, Latitude, Longitude, Age
lgl  (1): GenderNonconforming

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [47]:
df_new |>
#   mutate(StopDate = mdy(StopDate)) |>
#   summarise(min_date = min(StopDate), max_date = max(StopDate))
  distinct(TrafficViolation)
#   # filter(is.na(Longitude)) # because these are so few we could just exclude from the analysis for ease of analysis

TrafficViolation
<lgl>
NA


In [48]:
split_column = function(df, col){
    df |>
        filter(.data[[col]] != "None") |>
        # distinct(.data[[col]])
        pull(.data[[col]]) |>
        str_split("\\|") |>
        unlist() |>
        str_trim() |>
        unique()
}

In [49]:
new_nonforce = split_column(df_new, "NonForceActionsTaken")
new_force = split_column(df_new, "ForceActionsTaken")
old = split_column(df_old, "ActionsTaken")
old_unique = setdiff(old, new)

force_actions = c(new_force, old_unique[(!str_detect(old_unique, "search"))])
nonforce_actions = c(new_nonforce, old_unique[str_detect(old_unique, "search")])

df_old_actions = df_old |>
    separate_longer_delim(ActionsTaken, delim = "\\|") |>
    mutate(ActionsTaken = str_trim(ActionsTaken)) |>
    mutate(
        type = case_when(
            ActionsTaken %in% force_actions ~ "force",
            ActionsTaken %in% nonforce_actions ~ "nonforce",
        )
    ) |>
    group_by(LEA_RecordID, DateTimeOfStop) |>
    summarise(
        ForceActionsTaken = str_c(ActionsTaken[type == "force"], collapse = "|"),
        NonForceActionsTaken = str_c(ActionsTaken[type == "nonforce"], collapse = "|")
    ) |>
    mutate(
        NonForceActionsTaken = ifelse(NonForceActionsTaken %in% c("", NA), "None", NonForceActionsTaken),
        ForceActionsTaken = ifelse(ForceActionsTaken %in% c("", NA), "None", ForceActionsTaken),
        ) |>
    select(LEA_RecordID, NonForceActionsTaken, ForceActionsTaken)


In [50]:
# df_old |>
#   mutate(DateTimeOfStop = mdy_hm(DateTimeOfStop)) |>
#   summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))
  # filter(is.na(Longitude)) # none
  # distinct(TrafficViolation)

# names(df_old)

In [51]:
df_old_cleaned = df_old |>
  mutate(
    DateTimeOfStop = mdy_hm(DateTimeOfStop, tz = "America/Los_Angeles"),
    RespToServiceCall = ifelse(RespToServiceCall == "Yes", "Y", "N"),
    TypeOfStop = ifelse(
      TypeOfStop == "Vehicle", "Vehicular Stop", str_c(TypeOfStop, " Stop")
    ),
    LocationType = "LatLonPt",
  ) |>
  filter(
    # old data ends around mid-2024, exclude the 2024 data
    DateTimeOfStop >= start_date, year(DateTimeOfStop) <= 2023,
    City == "Berkeley",
  ) |>
  left_join(df_old_actions, by = "LEA_RecordID") |>
  select(LEA_Record_ID = LEA_RecordID, PersonNum, DateTimeOfStop,
    CallForService = RespToServiceCall, TypeOfStopDesc = TypeOfStop, StopDuration = DurationOfStop,
    City, LocationType, Latitude, Longitude, LocationDescription,
    NonForceActionsTaken, ForceActionsTaken
  )

df_old_cleaned |>
  summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))

head(df_old_cleaned, 5)

min_date,max_date
<dttm>,<dttm>
2021-03-01 00:03:00,2023-12-31 20:06:00


LEA_Record_ID,PersonNum,DateTimeOfStop,CallForService,TypeOfStopDesc,StopDuration,City,LocationType,Latitude,Longitude,LocationDescription,NonForceActionsTaken,ForceActionsTaken
<chr>,<dbl>,<dttm>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
BPD000000641,1,2021-03-01 00:03:00,N,Vehicular Stop,3,Berkeley,LatLonPt,37.868,-122.258,2500 block of Durant Ave,None,None
BPD000000646,1,2021-03-01 00:09:00,N,Pedestrian Stop,20,Berkeley,LatLonPt,37.869,-122.272,Civic center park,None,None
BPD000000642,1,2021-03-01 00:37:00,N,Bicycle Stop,5,Berkeley,LatLonPt,37.866,-122.301,2000 block of 2nd St,None,None
BPD000000643,1,2021-03-01 00:45:00,Y,Pedestrian Stop,25,Berkeley,LatLonPt,37.875,-122.270,2000 block of Delaware St,None,Handcuffed or flex cuffed
BPD000000649,1,2021-03-01 05:38:00,Y,Vehicular Stop,2,Berkeley,LatLonPt,37.871,-122.280,1900 block of California St,None,None


In [52]:
df_new_cleaned = df_new |>
  mutate(
    DateTimeOfStop = mdy_hms(str_c(StopDate, " ", StopTime), tz = "America/Los_Angeles"),
    City = str_to_title(City)
  ) |>
  filter(
    DateTimeOfStop <= end_date, # new data starts at 2024
    City == "Berkeley"
  ) |> 
  select(LEA_Record_ID, PersonNum, DateTimeOfStop,
    CallForService, TypeOfStopDesc, StopDuration,
    City, LocationType, Latitude, Longitude,
    BlockNumber, StreetName, CrossStreet1, CrossStreet2, Highway, ClosestExit, OtherLocationDesc,
    NonForceActionsTaken, ForceActionsTaken
  )

df_new_cleaned |>
  summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))


min_date,max_date
<dttm>,<dttm>
2024-01-01 01:20:00,2026-04-29 17:59:00


In [53]:
df = full_join(df_old_cleaned, df_new_cleaned) |>
    mutate(DateTimeOfStop = format(DateTimeOfStop, "%Y-%m-%d %H:%M:%S%z"))

write_csv(df, "../data/clean/stop_data.csv")

In [54]:
test = read_csv("../data/clean/stop_data.csv")
test

Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 24733 Columns: 20
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (9): LEA_Record_ID, CallForService, TypeOfStopDesc, City, LocationType,...
dbl  (5): PersonNum, StopDuration, Latitude, Longitude, BlockNumber
lgl  (5): CrossStreet1, CrossStreet2, Highway, ClosestExit, OtherLocationDesc
dttm (1): DateTimeOfStop

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


LEA_Record_ID,PersonNum,DateTimeOfStop,CallForService,TypeOfStopDesc,StopDuration,City,LocationType,Latitude,Longitude,LocationDescription,NonForceActionsTaken,ForceActionsTaken,BlockNumber,StreetName,CrossStreet1,CrossStreet2,Highway,ClosestExit,OtherLocationDesc
<chr>,<dbl>,<dttm>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
BPD000000641,1,2021-03-01 08:03:00,N,Vehicular Stop,3,Berkeley,LatLonPt,37.868,-122.258,2500 block of Durant Ave,None,None,NA,NA,NA,NA,NA,NA,NA
BPD000000646,1,2021-03-01 08:09:00,N,Pedestrian Stop,20,Berkeley,LatLonPt,37.869,-122.272,Civic center park,None,None,NA,NA,NA,NA,NA,NA,NA
BPD000000642,1,2021-03-01 08:37:00,N,Bicycle Stop,5,Berkeley,LatLonPt,37.866,-122.301,2000 block of 2nd St,None,None,NA,NA,NA,NA,NA,NA,NA
BPD000000643,1,2021-03-01 08:45:00,Y,Pedestrian Stop,25,Berkeley,LatLonPt,37.875,-122.270,2000 block of Delaware St,None,Handcuffed or flex cuffed,NA,NA,NA,NA,NA,NA,NA
BPD000000649,1,2021-03-01 13:38:00,Y,Vehicular Stop,2,Berkeley,LatLonPt,37.871,-122.280,1900 block of California St,None,None,NA,NA,NA,NA,NA,NA,NA
BPD000000650,1,2021-03-01 19:49:00,N,Vehicular Stop,7,Berkeley,LatLonPt,37.855,-122.239,Tunnel Rd,None,None,NA,NA,NA,NA,NA,NA,NA
BPD000000651,1,2021-03-01 20:11:00,N,Vehicular Stop,8,Berkeley,LatLonPt,37.855,-122.238,100 block of Tunnel Rd,None,None,NA,NA,NA,NA,NA,NA,NA
BPD000000652,1,2021-03-01 20:24:00,N,Vehicular Stop,8,Berkeley,LatLonPt,37.858,-122.242,0 block of Tunnel Rd,None,None,NA,NA,NA,NA,NA,NA,NA
BPD000000654,1,2021-03-01 21:00:00,N,Vehicular Stop,15,Berkeley,LatLonPt,37.869,-122.273,2100 block of Martin Luther King Jr Way,None,None,NA,NA,NA,NA,NA,NA,NA
